# Experiment 9: Feature Selection Methodology Analysis

This notebook analyzes results from Experiment 9, which addresses:
- **9A**: All-fields baseline (raw ETL output: ~82 fields ≥50% coverage, ~292 total)
- **9B**: Algorithmic selection comparison (8 methods from supervisor feedback)
- **9C**: Graph-only ablation (RQ1 gap)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
ARTIFACTS_DIR = Path('../artifacts/feature_selection')

# Load results
results_path = ARTIFACTS_DIR / 'selection_results.json'
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(f"Generated at: {results['generated_at']}")
    print(f"Available experiments: {list(results['results'].keys())}")
else:
    print(f"Results not found at {results_path}")
    print("Run: python -m src.experiments.exp9_feature_selection +exp9.method=full")
    results = None

## 9A: All-Fields Baseline

Tests performance using raw ETL output (`raw_insertions.parquet`) with:
- **~82 fields**: ≥50% coverage threshold
- **~292 fields**: All available fields (XGBoost handles missing values natively)

In [ ]:
if results and '9a_all_fields' in results['results']:
    r9a = results['results']['9a_all_fields']
    
    # Build comparison table
    rows = []
    for config_name, data in r9a.items():
        if isinstance(data, dict) and 'mean_auc_pr' in data:
            rows.append({
                'Configuration': config_name,
                'N Features': data['n_features'],
                'AUC-PR': data['mean_auc_pr'],
                'Std': data.get('std_auc_pr', 0)
            })
    
    if rows:
        df_9a = pd.DataFrame(rows)
        display(df_9a.style.format({'AUC-PR': '{:.4f}', 'Std': '{:.4f}'}))
        
        # Plot
        fig, ax = plt.subplots(figsize=(8, 4))
        bars = ax.bar(df_9a['Configuration'], df_9a['AUC-PR'], 
                      yerr=df_9a['Std'], capsize=5, color=['#3498db', '#2ecc71'])
        ax.set_ylabel('AUC-PR')
        ax.set_title('9A: All-Fields Baseline (Raw ETL Output)')
        ax.set_ylim(0, 1)
        for bar, n in zip(bars, df_9a['N Features']):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                    f'n={n}', ha='center', va='bottom', fontsize=10)
        plt.tight_layout()
        plt.show()
else:
    print("9A results not available. Run: +exp9.method=all_fields")

## 9B: Feature Selection Method Comparison

Compares 8 methods from supervisor feedback:

| Priority | Methods |
|----------|--------|
| HIGH | RFE, LASSO |
| MEDIUM | Information Gain, Chi-Square, Mutual Information, Permutation Importance |
| Baseline | Production (hand-picked), Correlation-filtered |

In [ ]:
if results and '9b_comparison' in results['results']:
    r9b = results['results']['9b_comparison']
    
    df_9b = pd.DataFrame(r9b)
    df_9b = df_9b.sort_values('mean_auc_pr', ascending=False)
    
    # Display table
    display(df_9b[['method', 'priority', 'n_features', 'mean_auc_pr', 'std_auc_pr']]
            .style.format({'mean_auc_pr': '{:.4f}', 'std_auc_pr': '{:.4f}'}))
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = {'HIGH': '#e74c3c', 'MEDIUM': '#f39c12', 'baseline': '#95a5a6'}
    bar_colors = [colors.get(p, '#3498db') for p in df_9b['priority']]
    
    bars = ax.barh(df_9b['method'], df_9b['mean_auc_pr'], 
                   xerr=df_9b['std_auc_pr'], capsize=3, color=bar_colors)
    ax.set_xlabel('AUC-PR')
    ax.set_title('9B: Feature Selection Method Comparison')
    ax.set_xlim(0, 1)
    
    # Add feature counts
    for bar, n in zip(bars, df_9b['n_features']):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                f'n={n}', va='center', fontsize=9)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=l) for l, c in colors.items()]
    ax.legend(handles=legend_elements, loc='lower right')
    
    plt.tight_layout()
    plt.show()
else:
    print("9B results not available. Run: +exp9.method=comparison")

## 9C: Graph-Only Ablation (RQ1 Gap)

Tests standalone predictive value of graph features vs tabular features.

In [ ]:
if results and '9c_graph_only' in results['results']:
    r9c = results['results']['9c_graph_only']
    
    graph_auc = r9c['graph_only']['mean_auc_pr']
    tabular_auc = r9c['tabular_only']['mean_auc_pr']
    graph_n = r9c['graph_only']['n_features']
    tabular_n = r9c['tabular_only']['n_features']
    
    print(f"Graph-only:   AUC-PR = {graph_auc:.4f} (n={graph_n} features)")
    print(f"Tabular-only: AUC-PR = {tabular_auc:.4f} (n={tabular_n} features)")
    print(f"Delta:        {graph_auc - tabular_auc:+.4f}")
    print()
    
    if graph_auc > 0.4:
        print("✓ Graph features HAVE standalone predictive value")
    else:
        print("⚠ Graph features may NOT have strong standalone value")
    
    # Plot
    fig, ax = plt.subplots(figsize=(6, 4))
    x = ['Graph-only', 'Tabular-only']
    y = [graph_auc, tabular_auc]
    colors = ['#9b59b6', '#3498db']
    bars = ax.bar(x, y, color=colors)
    ax.set_ylabel('AUC-PR')
    ax.set_title('9C: Graph vs Tabular Feature Ablation')
    ax.set_ylim(0, 1)
    for bar, val, n in zip(bars, y, [graph_n, tabular_n]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}\n(n={n})', ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("9C results not available. Run: +exp9.method=graph_only")

## Summary Table for Thesis

Consolidated results for Chapter 4 / Results section.

In [ ]:
if results:
    summary_rows = []
    
    # 9A results
    if '9a_all_fields' in results['results']:
        for config, data in results['results']['9a_all_fields'].items():
            if isinstance(data, dict) and 'mean_auc_pr' in data:
                summary_rows.append({
                    'Experiment': '9A',
                    'Method': config,
                    'N Features': data['n_features'],
                    'AUC-PR': data['mean_auc_pr']
                })
    
    # 9B results (top 3)
    if '9b_comparison' in results['results']:
        sorted_9b = sorted(results['results']['9b_comparison'], 
                          key=lambda x: x['mean_auc_pr'], reverse=True)
        for r in sorted_9b[:3]:
            summary_rows.append({
                'Experiment': '9B',
                'Method': f"{r['method']} ({r['priority']})",
                'N Features': r['n_features'],
                'AUC-PR': r['mean_auc_pr']
            })
    
    # 9C results
    if '9c_graph_only' in results['results']:
        r9c = results['results']['9c_graph_only']
        summary_rows.append({
            'Experiment': '9C',
            'Method': 'Graph-only',
            'N Features': r9c['graph_only']['n_features'],
            'AUC-PR': r9c['graph_only']['mean_auc_pr']
        })
        summary_rows.append({
            'Experiment': '9C',
            'Method': 'Tabular-only',
            'N Features': r9c['tabular_only']['n_features'],
            'AUC-PR': r9c['tabular_only']['mean_auc_pr']
        })
    
    if summary_rows:
        df_summary = pd.DataFrame(summary_rows)
        print("=" * 60)
        print("EXPERIMENT 9: FEATURE SELECTION SUMMARY")
        print("=" * 60)
        display(df_summary.style.format({'AUC-PR': '{:.4f}'}))
else:
    print("No results available yet.")

## Key Findings (Fill after running)

### 9A: All-Fields Baseline
- [ ] Compare ~82 fields (≥50% coverage) vs ~292 fields (all)
- [ ] Document if more fields = better performance or noise

### 9B: Feature Selection Methods
- [ ] Best performing method: ___
- [ ] Optimal feature count: ___
- [ ] HIGH priority methods (RFE, LASSO) outperform baseline: Yes/No

### 9C: Graph-Only Ablation
- [ ] Graph features alone AUC-PR: ___
- [ ] Standalone value confirmed: Yes/No
- [ ] Justifies hybrid approach: Yes/No